# Super Resolution SRGAN Pipeline for Remote Run
This notebook contains the complete pipeline to run Data Ingestion, Transformation, and Model Training sequentially on a remote machine like Colab.

In [ ]:
import sys
!{sys.executable} -m pip install h5py lpips tqdm scikit-image opencv-python tensorboard pyyaml matplotlib datasets torchvision

In [ ]:
import os
import urllib.request
import zipfile
import time
from pathlib import Path
from tqdm import tqdm
import cv2
import h5py
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models import vgg19, VGG19_Weights
import torchvision.utils as vutils
from skimage.metrics import peak_signal_noise_ratio as psnr_func
from skimage.metrics import structural_similarity as ssim_func
import lpips
import logging
import math

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s: %(message)s")
logger = logging.getLogger("remote_training")

In [ ]:
from dataclasses import dataclass

@dataclass
class DataIngestionConfig:
    root_dir: Path
    dataset_name: str
    dataset_root: Path
    train_hr_dir: Path
    train_lr_dir: Path
    val_hr_dir: Path
    val_lr_dir: Path
    test_dataset_name: str
    test_dataset_root: Path
    test_hr_dir: Path
    test_lr_dir: Path
    download_enabled: bool
    download_urls: dict
    processing_scale: int
    processing_patch_size: int
    processing_batch_size: int
    processing_num_workers: int

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    srcnn_dir: Path
    srgan_dir: Path
    patch_size: int
    stride: int
    scale: int

@dataclass
class SRGANTrainingConfig:
    root_dir: Path
    train_data_path: Path
    valid_data_path: Path
    model_path_g: Path
    model_path_d: Path
    pretrain_epochs: int
    epochs: int
    batch_size: int
    learning_rate_g: float
    learning_rate_d: float
    normalization: str
    device: str
    log_step: int
    patience: int

# Configuration Instances
ingestion_config = DataIngestionConfig(
    root_dir=Path("data/raw"),
    dataset_name="DIV2K",
    dataset_root=Path("data/raw/DIV2K"),
    train_hr_dir=Path("data/raw/DIV2K/DIV2K_train_HR"),
    train_lr_dir=Path("data/raw/DIV2K/DIV2K_train_LR_bicubic/X4"),
    val_hr_dir=Path("data/raw/DIV2K/DIV2K_valid_HR"),
    val_lr_dir=Path("data/raw/DIV2K/DIV2K_valid_LR_bicubic/X4"),
    test_dataset_name="Set5",
    test_dataset_root=Path("data/raw/Set5"),
    test_hr_dir=Path("data/raw/Set5/set5"),
    test_lr_dir=Path("data/raw/Set5/set5"),
    download_enabled=True,
    download_urls={
        "train_hr": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip",
        "train_lr_x4": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X4.zip",
        "val_hr": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip",
        "val_lr_x4": "https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip",
        "set5": "https://data.deepai.org/set5.zip"
    },
    processing_scale=4, processing_patch_size=96,
    processing_batch_size=16, processing_num_workers=4
)

transformation_config = DataTransformationConfig(
    root_dir=Path("data/processed"),
    data_path=Path("data/raw/DIV2K"),
    srcnn_dir=Path("data/processed/srcnn"),
    srgan_dir=Path("data/processed/srgan"),
    patch_size=96, stride=96, scale=4
)

training_config = SRGANTrainingConfig(
    root_dir=Path("artifacts/model_training"),
    train_data_path=Path("data/processed/srgan/srgan_train.h5"),
    valid_data_path=Path("data/processed/srgan/srgan_valid.h5"),
    model_path_g=Path("artifacts/model_training/srgan_gen.pth"),
    model_path_d=Path("artifacts/model_training/srgan_disc.pth"),
    pretrain_epochs=50, epochs=200, batch_size=16,
    learning_rate_g=0.0001, learning_rate_d=0.0001,
    normalization="minus_one_to_one", device="cuda",
    patience=7, log_step=100
)

for d in [ingestion_config.root_dir, transformation_config.root_dir, training_config.root_dir]:


In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def _data_exists(self) -> bool:
        required_paths = [self.config.train_hr_dir, self.config.val_hr_dir, self.config.test_hr_dir]
        status = True
        for p in required_paths:
            if not p.exists() or not any(p.iterdir()):
                status = False
        return status

    def download_file(self):
        if self._data_exists():
            logger.info("All required dataset directories exist and are populated. Skipping download phase.")
            return

        start_time = time.time()
        for name, url in self.config.download_urls.items():
            zip_path = Path(self.config.root_dir) / f"{name}.zip"
            extract_to = self.config.test_dataset_root if name == "set5" else self.config.dataset_root
            if name == "set5": check_dir = self.config.test_hr_dir
            elif name == "train_hr": check_dir = self.config.train_hr_dir
            elif name == "train_lr_x4": check_dir = self.config.train_lr_dir
            elif name == "val_hr": check_dir = self.config.val_hr_dir
            elif name == "val_lr_x4": check_dir = self.config.val_lr_dir
            else: check_dir = self.config.train_hr_dir
            if check_dir.exists() and any(check_dir.iterdir()):
                logger.info(f"Skipping {name}: Data already exists.")
                continue

            try:
                os.makedirs(extract_to, exist_ok=True)
                self._download_with_progress(url, zip_path, name)
                self._perform_extraction(zip_path, extract_to)
                logger.info(f"Successfully processed {name}")
            except Exception as e:
                logger.error(f"Error processing {name}: {e}")
                if zip_path.exists(): os.remove(zip_path)

    def _perform_extraction(self, zip_path: Path, extract_to: Path):
        logger.info(f"Extracting {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        if zip_path.exists(): os.remove(zip_path)

        for root, dirs, files in os.walk(extract_to):
            for file in files:
                file_path = Path(root) / file
                if file.endswith('.zip'):
                    logger.info(f"Found nested zip: {file}. Extracting...")
                    with zipfile.ZipFile(file_path, 'r') as nested_ref:
                        nested_ref.extractall(root)
                    os.remove(file_path)
                elif file.endswith('.url') or 'Startcrack' in file:
                    os.remove(file_path)

    def _download_with_progress(self, url: str, filepath: Path, name: str):
        headers = {'User-Agent': 'Mozilla/5.0'}
        request = urllib.request.Request(url, headers=headers)
        with tqdm(unit='B', unit_scale=True, unit_divisor=1024, miniters=1, desc=f"Downloading {name}") as t:
            with urllib.request.urlopen(request) as response, open(filepath, 'wb') as out_file:
                tsize = int(response.info().get('Content-Length', 0))
                t.total = tsize
                bsize = 1024 * 8
                while True:
                    block = response.read(bsize)
                    if not block: break
                    out_file.write(block)
                    t.update(len(block))

In [ ]:
class DataTransformation:
    def __init__(self, config):
        self.config = config

    def create_srgan_data(self, split='train'):
        hr_path = os.path.join(self.config.data_path, f"DIV2K_{split}_HR")
        lr_path = os.path.join(self.config.data_path, f"DIV2K_{split}_LR_bicubic", "X4")
        
        save_path = Path(self.config.srgan_dir) / f"srgan_{split}.h5"
        save_path.parent.mkdir(parents=True, exist_ok=True)

        scale = self.config.scale
        lr_patch_size = self.config.patch_size // scale

        with h5py.File(save_path, 'w') as h5_file:
            lr_ds = h5_file.create_dataset("lr", shape=(0, lr_patch_size, lr_patch_size, 3), 
                                          maxshape=(None, lr_patch_size, lr_patch_size, 3),
                                          dtype='uint8', chunks=True)
            hr_ds = h5_file.create_dataset("hr", shape=(0, self.config.patch_size, self.config.patch_size, 3), 
                                          maxshape=(None, self.config.patch_size, self.config.patch_size, 3),
                                          dtype='uint8', chunks=True)

            img_list = os.listdir(hr_path)
            total_patches = 0

            for i, img_name in enumerate(img_list):
                hr_img = cv2.imread(os.path.join(hr_path, img_name))
                lr_img = cv2.imread(os.path.join(lr_path, img_name.replace(".png", "x4.png")))
                if hr_img is None or lr_img is None: continue

                current_img_patches_lr = []
                current_img_patches_hr = []

                for y in range(0, hr_img.shape[0] - self.config.patch_size + 1, self.config.stride):
                    for x in range(0, hr_img.shape[1] - self.config.patch_size + 1, self.config.stride):
                        hr_patch = hr_img[y:y+self.config.patch_size, x:x+self.config.patch_size]
                        lr_patch = lr_img[y//scale:(y+self.config.patch_size)//scale, x//scale:(x+self.config.patch_size)//scale]
                        
                        current_img_patches_hr.append(hr_patch)
                        current_img_patches_lr.append(lr_patch)

                if current_img_patches_hr:
                    num_new_patches = len(current_img_patches_hr)
                    lr_ds.resize(total_patches + num_new_patches, axis=0)
                    hr_ds.resize(total_patches + num_new_patches, axis=0)
                    lr_ds[total_patches:] = np.array(current_img_patches_lr)
                    hr_ds[total_patches:] = np.array(current_img_patches_hr)
                    total_patches += num_new_patches

                if i % 10 == 0:
                    logger.info(f"Processed {i}/{len(img_list)} images. Total patches: {total_patches}")

        logger.info(f"Final dataset saved. Total patches: {total_patches}")


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x): return x + self.bn2(self.conv2(self.prelu(self.bn1(self.conv1(x)))))

class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, up_scale):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * up_scale ** 2, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(up_scale)
        self.prelu = nn.PReLU()
    def forward(self, x): return self.prelu(self.pixel_shuffle(self.conv(x)))

class Generator(nn.Module):
    def __init__(self, scale_factor=4, num_residual_blocks=16):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 9, 1, 4)
        self.prelu = nn.PReLU()
        self.res_blocks = nn.Sequential(*[ResidualBlock(64) for _ in range(num_residual_blocks)])
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(64)
        self.upsample_blocks = nn.Sequential(*[UpsampleBlock(64, 2) for _ in range(int(math.log2(scale_factor)))])
        self.conv3 = nn.Conv2d(64, 3, 9, 1, 4)

    def forward(self, x):
        out1 = self.prelu(self.conv1(x))
        out = self.res_blocks(out1)
        out2 = self.bn2(self.conv2(out))
        out = self.upsample_blocks(out1 + out2)
        return torch.tanh(self.conv3(out))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(512, 1024, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(1024, 1, 1)
        )
    def forward(self, x): return self.net(x).view(x.size(0))

class VGGLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.vgg = vgg19(weights=VGG19_Weights.DEFAULT).features[:36].eval().to(device)
        self.loss = nn.MSELoss()
        for param in self.vgg.parameters(): param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device))

    def forward(self, input, target):
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std
        return self.loss(self.vgg(input), self.vgg(target))

class EdgeLoss(nn.Module):
    def __init__(self, device):
        super(EdgeLoss, self).__init__()
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [ 0.,  0.,  0.], [ 1.,  2.,  1.]], dtype=torch.float32).view(1, 1, 3, 3)
        self.sobel_x = sobel_x.repeat(3, 1, 1, 1).to(device)
        self.sobel_y = sobel_y.repeat(3, 1, 1, 1).to(device)
        self.l1_loss = nn.L1Loss()

    def forward(self, pred, target):
        pred_x = F.conv2d(pred, self.sobel_x, padding=1, groups=3)
        pred_y = F.conv2d(pred, self.sobel_y, padding=1, groups=3)
        target_x = F.conv2d(target, self.sobel_x, padding=1, groups=3)
        target_y = F.conv2d(target, self.sobel_y, padding=1, groups=3)
        pred_mag = torch.sqrt(pred_x**2 + pred_y**2 + 1e-6)
        target_mag = torch.sqrt(target_x**2 + target_y**2 + 1e-6)
        return self.l1_loss(pred_mag, target_mag)

In [ ]:
class HDF5Dataset(Dataset):
    def __init__(self, h5_file, normalization='minus_one_to_one'):
        super().__init__()
        self.h5_file = h5_file; self.norm = normalization

    def __getitem__(self, index):
        with h5py.File(self.h5_file, 'r') as f:
            lr = np.array(f['lr'][index]).astype(np.float32)
            hr = np.array(f['hr'][index]).astype(np.float32)
            if self.norm == "minus_one_to_one":
                lr = (lr / 127.5) - 1.0; hr = (hr / 127.5) - 1.0
            return torch.from_numpy(lr).permute(2, 0, 1), torch.from_numpy(hr).permute(2, 0, 1)

    def __len__(self):
        with h5py.File(self.h5_file, 'r') as f: return len(f['lr'])

loss_fn_alex = lpips.LPIPS(net='alex')

def calculate_psnr(img1, img2): return psnr_func(img1, img2, data_range=255.0)
def calculate_ssim(img1, img2): return ssim_func(img1, img2, channel_axis=2, data_range=255.0)
def calculate_lpips(img1_tensor, img2_tensor):
    with torch.no_grad():
        if len(img1_tensor.shape) == 3: img1_tensor, img2_tensor = img1_tensor.unsqueeze(0), img2_tensor.unsqueeze(0)
        dist = loss_fn_alex(img1_tensor.cpu(), img2_tensor.cpu())
    return dist.mean().item()
def calculate_edge_fidelity(img1, img2):
    grad1_x = cv2.Sobel(img1, cv2.CV_64F, 1, 0, ksize=3); grad1_y = cv2.Sobel(img1, cv2.CV_64F, 0, 1, ksize=3)
    grad2_x = cv2.Sobel(img2, cv2.CV_64F, 1, 0, ksize=3); grad2_y = cv2.Sobel(img2, cv2.CV_64F, 0, 1, ksize=3)
    mag1 = np.sqrt(grad1_x**2 + grad1_y**2); mag2 = np.sqrt(grad2_x**2 + grad2_y**2)
    return np.mean((mag1 - mag2)**2)

In [ ]:
class SRGANTraining:
    def __init__(self, config: SRGANTrainingConfig):
        self.config = config
        self.device = torch.device(config.device if torch.cuda.is_available() else "cpu")
        self.writer = SummaryWriter(log_dir=str(self.config.root_dir / "logs_srgan"))
        self.scaler_g = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
        self.scaler_d = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    def train(self):
        train_loader = self._get_dataloader(self.config.train_data_path)
        valid_loader = self._get_dataloader(self.config.valid_data_path)

        netG = Generator().to(self.device); netD = Discriminator().to(self.device)
        vgg_loss = VGGLoss(self.device); mse_loss = nn.MSELoss()
        bce_loss = nn.BCEWithLogitsLoss(); edge_loss = EdgeLoss(self.device).to(self.device)
        
        optimizer_g = optim.Adam(netG.parameters(), lr=self.config.learning_rate_g)
        optimizer_d = optim.Adam(netD.parameters(), lr=self.config.learning_rate_d)
        best_psnr = 0.0; global_step = 0

        if os.path.exists(self.config.model_path_g) and os.path.exists(self.config.model_path_d):
            logger.info("Loading existing checkpoints")
            netG.load_state_dict(torch.load(self.config.model_path_g, map_location=self.device))
            netD.load_state_dict(torch.load(self.config.model_path_d, map_location=self.device))


        if not os.path.exists(self.config.model_path_g) and self.config.pretrain_epochs > 0:
            logger.info(f"Starting Generator Pre-training ({self.config.pretrain_epochs} epochs) with MSE Loss only on {self.device}")
            for pretrain_epoch in range(self.config.pretrain_epochs):
                netG.train()
                epoch_loss_g = 0.0
                progress_bar = tqdm(train_loader, desc=f"Pre-Train Epoch {pretrain_epoch+1}/{self.config.pretrain_epochs}")
                for batch_idx, (lr, hr) in enumerate(progress_bar):
                    lr, hr = lr.to(self.device), hr.to(self.device)
                    optimizer_g.zero_grad()
                    amp_ctx = torch.amp.autocast('cuda') if self.scaler_g else torch.autocast("cpu", enabled=False)
                    with amp_ctx:
                        fake_hr = netG(lr)
                        g_loss = mse_loss(fake_hr, hr)
                    if self.scaler_g:
                        self.scaler_g.scale(g_loss).backward(); self.scaler_g.step(optimizer_g); self.scaler_g.update()
                    else:
                        g_loss.backward(); optimizer_g.step()
                        
                    epoch_loss_g += g_loss.item()
                    global_step += 1
                    if global_step % self.config.log_step == 0:
                        progress_bar.set_postfix({"G_MSE": f"{epoch_loss_g/(batch_idx+1):.4f}"})
                        
                val_results = self._validate(netG, valid_loader, mse_loss, epoch=None)
                avg_psnr, avg_ssim = val_results[1], val_results[2]
                logger.info(f"Pre-Train Epoch {pretrain_epoch+1}: PSNR={avg_psnr:.2f}dB | SSIM={avg_ssim:.4f}")
            
            torch.save(netG.state_dict(), self.config.model_path_g)
            global_step = 0

        for epoch in range(self.config.epochs):
            netG.train(); netD.train(); epoch_loss_g, epoch_loss_d = 0.0, 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs}")
            
            for batch_idx, (lr, hr) in enumerate(progress_bar):
                lr, hr = lr.to(self.device), hr.to(self.device)
                
                optimizer_d.zero_grad()
                amp_ctx = torch.amp.autocast('cuda') if self.scaler_d else torch.autocast("cpu", enabled=False)
                with amp_ctx:
                    fake_hr = netG(lr)
                    real_pred = netD(hr); fake_pred = netD(fake_hr.detach())
                    d_loss = bce_loss(real_pred, torch.ones_like(real_pred)) + bce_loss(fake_pred, torch.zeros_like(fake_pred))
                if self.scaler_d: self.scaler_d.scale(d_loss).backward(); self.scaler_d.step(optimizer_d); self.scaler_d.update()
                else: d_loss.backward(); optimizer_d.step()
                
                optimizer_g.zero_grad()
                amp_ctx = torch.amp.autocast('cuda') if self.scaler_g else torch.autocast("cpu", enabled=False)
                with amp_ctx:
                    fake_pred = netD(fake_hr)
                    g_adv_loss = bce_loss(fake_pred, torch.ones_like(fake_pred))
                    g_content_loss = vgg_loss((fake_hr+1)/2, (hr+1)/2)
                    g_edge_loss = edge_loss(fake_hr, hr)
                    g_mse_loss = mse_loss(fake_hr, hr)
                    g_loss = g_mse_loss + 0.006 * g_content_loss + 1e-3 * g_adv_loss + 0.1 * g_edge_loss
                if self.scaler_g: self.scaler_g.scale(g_loss).backward(); self.scaler_g.step(optimizer_g); self.scaler_g.update()
                else: g_loss.backward(); optimizer_g.step()
                
                epoch_loss_d += d_loss.item(); epoch_loss_g += g_loss.item(); global_step += 1
                
                if global_step % self.config.log_step == 0:
                    avg_d = epoch_loss_d/(batch_idx+1); avg_g = epoch_loss_g/(batch_idx+1)
                    self.writer.add_scalar("Loss/D_Step", avg_d, global_step)
                    self.writer.add_scalar("Loss/G_Step", avg_g, global_step)
                    progress_bar.set_postfix({"D": f"{avg_d:.4f}", "G": f"{avg_g:.4f}"})
            
            avg_val_loss, avg_psnr, avg_ssim, avg_lpips, avg_gms = self._validate(netG, valid_loader, mse_loss, epoch)
            self.writer.add_scalar("Loss/Validation_MSE", avg_val_loss, epoch)
            self.writer.add_scalar("Metrics/PSNR_dB", avg_psnr, epoch)
            self.writer.add_scalar("Metrics/SSIM", avg_ssim, epoch)
            self.writer.add_scalar("Metrics/LPIPS", avg_lpips, epoch)
            self.writer.add_scalar("Metrics/GMS", avg_gms, epoch)
            logger.info(f"Epoch {epoch+1}: PSNR={avg_psnr:.2f}dB | SSIM={avg_ssim:.4f} | LPIPS={avg_lpips:.4f} | GMS={avg_gms:.4f}")
            
            if avg_psnr > best_psnr:
                best_psnr = avg_psnr
                torch.save(netG.state_dict(), self.config.model_path_g)
                torch.save(netD.state_dict(), self.config.model_path_d)
        self.writer.close()

    def _validate(self, netG, loader, criterion, epoch=None):
        netG.eval(); val_loss = 0.0; psnr_values, ssim_values, lpips_values, gms_values = [], [], [], []
        with torch.no_grad():
            for batch_idx, (lr, hr) in enumerate(tqdm(loader, desc="Validation")):
                lr, hr = lr.to(self.device), hr.to(self.device)
                amp_ctx = torch.amp.autocast('cuda') if torch.cuda.is_available() else torch.autocast("cpu", enabled=False)
                with amp_ctx:
                    fake_hr = netG(lr)
                    loss = criterion(fake_hr, hr)
                val_loss += loss.item()
                
                if batch_idx == 0 and epoch is not None:
                    lr_up = torch.nn.functional.interpolate(lr, size=(hr.shape[2], hr.shape[3]), mode='bicubic', align_corners=False)
                    vis_lr = (lr_up[:4] + 1) / 2.0; vis_hr = (hr[:4] + 1) / 2.0; vis_fake = (fake_hr[:4] + 1) / 2.0
                    grid = vutils.make_grid(torch.cat([vis_lr, vis_fake, vis_hr], dim=0), nrow=4, normalize=False)
                    self.writer.add_image("Inference_Visibility/LR_FakeHR_RealHR", grid, epoch)

                out_np = np.clip((fake_hr.cpu().numpy() + 1.0) * 127.5, 0, 255).astype(np.uint8)
                hr_np = np.clip((hr.cpu().numpy() + 1.0) * 127.5, 0, 255).astype(np.uint8)
                
                for i in range(out_np.shape[0]):
                    img_out = np.transpose(out_np[i], (1, 2, 0)); img_hr = np.transpose(hr_np[i], (1, 2, 0))
                    psnr_values.append(calculate_psnr(img_out, img_hr))
                    ssim_values.append(calculate_ssim(img_out, img_hr))
                    lpips_values.append(calculate_lpips(fake_hr[i:i+1], hr[i:i+1]))
                    img_out_gray = cv2.cvtColor(img_out, cv2.COLOR_RGB2GRAY); img_hr_gray = cv2.cvtColor(img_hr, cv2.COLOR_RGB2GRAY)
                    gms_values.append(calculate_edge_fidelity(img_out_gray, img_hr_gray))
                    
        return val_loss / len(loader), np.mean(psnr_values), np.mean(ssim_values), np.mean(lpips_values), np.mean(gms_values)

    def _get_dataloader(self, path):
        dataset = HDF5Dataset(path, normalization=self.config.normalization)


In [ ]:
logger.info("--- Starting Data Ingestion ---")
di = DataIngestion(ingestion_config)
di.download_file()

In [ ]:
logger.info("--- Starting Data Transformation ---")
dt = DataTransformation(transformation_config)
if not os.path.exists(transformation_config.srgan_dir / "srgan_train.h5"):
    logger.info("Creating Train HR/LR patches...")
    dt.create_srgan_data('train')
else: logger.info("Train HDF5 exists. Skipping.")

if not os.path.exists(transformation_config.srgan_dir / "srgan_valid.h5"):
    logger.info("Creating Valid HR/LR patches...")
    dt.create_srgan_data('valid')
else: logger.info("Valid HDF5 exists. Skipping.")

In [ ]:
logger.info("--- Starting SRGAN Model Training ---")
mt = SRGANTraining(training_config)
mt.train()